In [ ]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.171  Python-3.14.3 torch-2.14.1+cu132 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
Setup complete  (16 CPUs, 63.7 GB RAM, 343.1/474.7 GB disk)


In [3]:
import torch

print(f"PyTorch version: {torch.version}") 
cuda_available = torch.cuda.is_available() 
print(f"CUDA available: {cuda_available}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
print(f"Using device: {device}")

PyTorch version: <module 'torch.version' from 'c:\\Users\\USER\\Desktop\\uni\\semestre 9\\vision por computadora\\entregable 1\\vision-por-computador\\.venv\\Lib\\site-packages\\torch\\version.py'>
CUDA available: True
Using device: cuda


In [9]:

import os
import random
import cv2
import xml.etree.ElementTree as ET
import numpy as np
from ultralytics import SAM
import gc

In [ ]:
model = SAM("sam_b.pt")
model.info()
dataset_base_path = os.path.join(os.path.expanduser('~') ,"Downloads/MASATI-v2")
SAM_path = os.path.join(os.path.expanduser('~') ,"Downloads/MASATI-v2/AutoSegmentacion")
ships_path = os.path.join(os.path.expanduser('~') ,"Downloads/MASATI-v2/ship")
ships_data = os.listdir(ships_path)

for _ in range(20):
    image = random.choice(ships_data)

    results = model(os.path.join(ships_path, image))

    img_SAM = results[0].plot()
    img = cv2.imread(os.path.join(ships_path, image))

    img_CONCAT = cv2.hconcat([img_SAM,img])

    cv2.imwrite(os.path.join(SAM_path, image[:-4]+"_SAM.png"), img_CONCAT)

gc.collect()
torch.cuda.empty_cache()


Model summary: 178 layers, 93,735,472 parameters, 93,735,472 gradients

image 1/1 C:\Users\USER\Downloads\MASATI-v2\ship\s0042.png: 1024x1024 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 3812.8ms
Speed: 5.3ms preprocess, 3812.8ms inference, 1.5ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 C:\Users\USER\Downloads\MASATI-v2\ship\s0419.png: 1024x1024 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 1 10, 1 11, 1 12, 1 13, 1 14, 1 15, 1 16, 3913.0ms
Speed: 5.3ms preprocess, 3913.0ms inference, 2.2ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 C:\Users\USER\Downloads\MASATI-v2\ship\s0918.png: 1024x1024 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 4459.4ms
Speed: 5.6ms preprocess, 4459.4ms inference, 1.8ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 C:\Users\USER\Downloads\MASATI-v2\ship\s0273.png: 1024x1024 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 1 10, 1 11, 1 12, 4014.4ms
Speed: 7.0ms preprocess, 4014.4ms inference, 1

In [ ]:
def save_voc_xml(results, image_path, xml_path, class_name="ship"):
    h, w = results.orig_shape            # (alto, ancho) de la imagen original
    boxes = results.boxes.xyxy.cpu().numpy()

    root = ET.Element("annotation")
    ET.SubElement(root, "folder").text = os.path.basename(os.path.dirname(image_path))
    ET.SubElement(root, "filename").text = os.path.basename(image_path)

    size = ET.SubElement(root, "size")
    ET.SubElement(size, "width").text = str(w)
    ET.SubElement(size, "height").text = str(h)
    ET.SubElement(size, "depth").text = "3"

    for x1, y1, x2, y2 in boxes:
        obj = ET.SubElement(root, "object")
        ET.SubElement(obj, "name").text = class_name
        ET.SubElement(obj, "pose").text = "Unspecified"
        ET.SubElement(obj, "truncated").text = "0"
        ET.SubElement(obj, "difficult").text = "0"

        bnd = ET.SubElement(obj, "bndbox")
        ET.SubElement(bnd, "xmin").text = str(int(round(x1)))
        ET.SubElement(bnd, "ymin").text = str(int(round(y1)))
        ET.SubElement(bnd, "xmax").text = str(int(round(x2)))
        ET.SubElement(bnd, "ymax").text = str(int(round(y2)))

    tree = ET.ElementTree(root)
    ET.indent(tree, space="  ")          
    tree.write(xml_path, encoding="utf-8", xml_declaration=True)

dataset_base_path = os.path.join(os.path.expanduser('~') ,"Downloads/MASATI-v2")
xml_path = os.path.join(os.path.expanduser('~') ,"Downloads/MASATI-v2/ships_SAM")
masati_content = os.listdir(dataset_base_path)
with_labels = {}

# find folders with labels
for folder in masati_content: 
    if folder.endswith("_labels"):
        with_labels[folder.replace("_labels", "")] = folder


for _ in range(7):
    for data, label in with_labels.items():
        data_path = os.path.join(dataset_base_path, data)
        label_path = os.path.join(dataset_base_path, label)
        image = random.choice(os.listdir(data_path))

        tree = ET.parse(os.path.join(label_path, image.replace(".png", ".xml")))
        root = tree.getroot()
        rectangles = []
        for obj in root.findall("object"):
            bbox = obj.find("bndbox")

            xmin = int(bbox.find("xmin").text)
            ymin = int(bbox.find("ymin").text)
            xmax = int(bbox.find("xmax").text)
            ymax = int(bbox.find("ymax").text)

            rectangles.append((xmin, ymin, xmax, ymax))

        results = model(os.path.join(data_path, image), bboxes=rectangles)

        save_voc_xml(results[0], os.path.join(data_path, image), os.path.join(xml_path,image[:-4]+".xml"), class_name="ship")


gc.collect()
torch.cuda.empty_cache()



image 1/1 C:\Users\USER\Downloads\MASATI-v2\coast_ship\x0227.png: 1024x1024 1 0, 479.1ms
Speed: 9.1ms preprocess, 479.1ms inference, 1.7ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 C:\Users\USER\Downloads\MASATI-v2\multi\m0106.png: 1024x1024 1 0, 1 1, 1 2, 231.1ms
Speed: 4.8ms preprocess, 231.1ms inference, 1.6ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 C:\Users\USER\Downloads\MASATI-v2\ship\s0025.png: 1024x1024 1 0, 226.3ms
Speed: 5.3ms preprocess, 226.3ms inference, 1.2ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 C:\Users\USER\Downloads\MASATI-v2\coast_ship\x0370.png: 1024x1024 1 0, 229.4ms
Speed: 5.7ms preprocess, 229.4ms inference, 1.2ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 C:\Users\USER\Downloads\MASATI-v2\multi\m0021.png: 1024x1024 1 0, 1 1, 1 2, 227.9ms
Speed: 5.5ms preprocess, 227.9ms inference, 1.2ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 C:\Users\USER\Downloads\MASATI-v2\s

In [21]:
dataset_base_path = os.path.join(os.path.expanduser('~') ,"Downloads/MASATI-v2")
SAM_path = os.path.join(dataset_base_path, "ships_SAM")
SAM_content = os.listdir(SAM_path)

for file in SAM_content:

    tree = ET.parse(os.path.join(SAM_path, file))
    root = tree.getroot()

    image = os.path.join(dataset_base_path, root.find('folder').text, root.find('filename').text)

    rectangles_SAM = []
    for obj in root.findall("object"):
        bbox = obj.find("bndbox")

        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        rectangles_SAM.append((xmin, ymin, xmax, ymax))


    cv2_img = cv2.imread(image)
    img_SAM = cv2_img.copy()
    for (x1,y1,x2,y2) in rectangles_SAM:
        cv2.rectangle(img_SAM, (x1,y1), (x2, y2), (0,0,255), 1)

    tree = ET.parse(os.path.join(dataset_base_path, root.find('folder').text+"_labels", root.find('filename').text.replace(".png", ".xml")))
    root = tree.getroot()
    rectangles_xml = []
    for obj in root.findall("object"):
        bbox = obj.find("bndbox")

        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        rectangles_xml.append((xmin, ymin, xmax, ymax))

    img_xml = cv2_img.copy()
    for (x1,y1,x2,y2) in rectangles_xml:
        cv2.rectangle(img_xml, (x1,y1), (x2, y2), (0,0,255), 1)

    img_CONCAT = cv2.hconcat([img_SAM,img_xml])

    cv2.imshow(image, img_CONCAT)
    cv2.waitKey(0)
    cv2.destroyAllWindows()